In [7]:
# ============================================================
# HSRIS — Part 1: Categorical Foundation (The Encoders)
# Built from first principles. No scikit-learn.
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ────────────────────────────────────────────────────────────
# 1. DATA LOADING
# ────────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/suraj520/customer-support-ticket-dataset"

# Auto-discover the CSV inside the dataset folder
csv_files = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"No CSV found under {DATA_DIR}")
csv_path = csv_files[0]
print(f"[INFO] Loading dataset from: {csv_path}")

df = pd.read_csv(csv_path)
print(f"[INFO] Dataset shape: {df.shape}")
print(f"[INFO] Columns: {list(df.columns)}\n")


# ────────────────────────────────────────────────────────────
# 2. CUSTOM LABEL ENCODER  (ordinal mapping with <UNK>)
# ────────────────────────────────────────────────────────────
class CustomLabelEncoder:
    """
    Maps categorical string labels → ordinal integers.
    Supports an explicit ordered mapping *or* auto-fit from data.
    Unseen categories at transform time are mapped to a dedicated
    <UNK> index so inference never crashes on new values.
    """

    def __init__(self, ordered_labels: list[str] | None = None):
        """
        Parameters
        ----------
        ordered_labels : list[str], optional
            If supplied, the encoder uses this exact ordering
            (e.g. ["Low", "Medium", "High"] → 0, 1, 2).
            If None, ordering is inferred alphabetically during fit().
        """
        self._ordered_labels = ordered_labels
        self.label_to_idx: dict[str, int] = {}
        self.idx_to_label: dict[int, str] = {}
        self.unk_idx: int = -1          # will be set in fit()
        self._is_fitted: bool = False

    def fit(self, values: pd.Series) -> "CustomLabelEncoder":
        """Learn the mapping from the training data."""
        if self._ordered_labels is not None:
            categories = self._ordered_labels
        else:
            categories = sorted(values.dropna().unique().tolist())

        self.label_to_idx = {label: idx for idx, label in enumerate(categories)}
        self.idx_to_label = {idx: label for label, idx in self.label_to_idx.items()}

        # Reserve the next integer for <UNK>
        self.unk_idx = len(categories)
        self.label_to_idx["<UNK>"] = self.unk_idx
        self.idx_to_label[self.unk_idx] = "<UNK>"

        self._is_fitted = True
        return self

    def transform(self, values: pd.Series) -> np.ndarray:
        """Convert labels → integers, mapping unknowns to unk_idx."""
        if not self._is_fitted:
            raise RuntimeError("Encoder has not been fitted yet.")
        return np.array(
            [self.label_to_idx.get(str(v), self.unk_idx) for v in values],
            dtype=np.int64,
        )

    def fit_transform(self, values: pd.Series) -> np.ndarray:
        return self.fit(values).transform(values)

    def inverse_transform(self, indices: np.ndarray) -> list[str]:
        """Map integers back to labels."""
        return [self.idx_to_label.get(int(i), "<UNK>") for i in indices]

    def __repr__(self) -> str:
        return (
            f"CustomLabelEncoder(classes={list(self.label_to_idx.keys())}, "
            f"unk_idx={self.unk_idx})"
        )


# ────────────────────────────────────────────────────────────
# 3. CUSTOM ONE-HOT ENCODER  (binary vector with <UNK> safety)
# ────────────────────────────────────────────────────────────
class CustomOneHotEncoder:
    """
    Converts a categorical column into a binary (one-hot) NumPy matrix.
    Unseen categories at transform time produce an all-zeros row
    instead of raising a KeyError.
    """

    def __init__(self):
        self.categories: list[str] = []
        self.cat_to_idx: dict[str, int] = {}
        self._is_fitted: bool = False

    def fit(self, values: pd.Series) -> "CustomOneHotEncoder":
        """Learn the unique categories from the training data."""
        self.categories = sorted(values.dropna().unique().tolist())
        self.cat_to_idx = {cat: idx for idx, cat in enumerate(self.categories)}
        self._is_fitted = True
        return self

    def transform(self, values: pd.Series) -> np.ndarray:
        """
        Encode each value as a one-hot row vector.

        Returns
        -------
        np.ndarray of shape (n_samples, n_categories)
            All-zeros row for any unseen / NaN category.
        """
        if not self._is_fitted:
            raise RuntimeError("Encoder has not been fitted yet.")

        n = len(values)
        k = len(self.categories)
        matrix = np.zeros((n, k), dtype=np.float32)

        for i, v in enumerate(values):
            idx = self.cat_to_idx.get(str(v))
            if idx is not None:
                matrix[i, idx] = 1.0
        return matrix

    def fit_transform(self, values: pd.Series) -> np.ndarray:
        return self.fit(values).transform(values)

    def get_feature_names(self) -> list[str]:
        """Column names useful for DataFrame conversion."""
        return [f"channel_{cat}" for cat in self.categories]

    def __repr__(self) -> str:
        return f"CustomOneHotEncoder(categories={self.categories})"


# ────────────────────────────────────────────────────────────
# 4. EXECUTION — apply both encoders & verify
# ────────────────────────────────────────────────────────────

# --- Label Encoder for Ticket Priority ---
priority_encoder = CustomLabelEncoder(ordered_labels=["Low", "Medium", "High"])
df["Priority_Encoded"] = priority_encoder.fit_transform(df["Ticket Priority"])

print("=== Custom Label Encoder ===")
print(priority_encoder)
print(f"Mapping: {priority_encoder.label_to_idx}\n")

# --- One-Hot Encoder for Ticket Channel ---
channel_encoder = CustomOneHotEncoder()
onehot_matrix = channel_encoder.fit_transform(df["Ticket Channel"])

# Attach each one-hot column to the DataFrame
for col_idx, col_name in enumerate(channel_encoder.get_feature_names()):
    df[col_name] = onehot_matrix[:, col_idx].astype(int)

print("=== Custom One-Hot Encoder ===")
print(channel_encoder)
print(f"Feature names: {channel_encoder.get_feature_names()}\n")

# --- Print first 5 rows to verify ---
display_cols = (
    ["Ticket Priority", "Priority_Encoded", "Ticket Channel"]
    + channel_encoder.get_feature_names()
)
print("=== First 5 Rows (Verification) ===")
print(df[display_cols].head().to_string(index=False))

# ────────────────────────────────────────────────────────────
# 5. SMOKE TEST — <UNK> / unseen-category safety
# ────────────────────────────────────────────────────────────
print("\n=== Unseen-Category Smoke Test ===")

# Label encoder: unseen priority
test_priorities = pd.Series(["High", "Critical", "Low"])
encoded = priority_encoder.transform(test_priorities)
print(f"Input:   {test_priorities.tolist()}")
print(f"Encoded: {encoded.tolist()}   (Note: 'Critical' → {priority_encoder.unk_idx} = <UNK>)")

# One-hot encoder: unseen channel
test_channels = pd.Series(["Email", "Carrier Pigeon", "Chat"])
onehot = channel_encoder.transform(test_channels)
print(f"\nInput:   {test_channels.tolist()}")
print(f"One-Hot:\n{onehot}   (Note: 'Carrier Pigeon' → all-zeros row)")


[INFO] Loading dataset from: /kaggle/input/datasets/suraj520/customer-support-ticket-dataset/customer_support_tickets.csv
[INFO] Dataset shape: (8469, 17)
[INFO] Columns: ['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

=== Custom Label Encoder ===
CustomLabelEncoder(classes=['Low', 'Medium', 'High', '<UNK>'], unk_idx=3)
Mapping: {'Low': 0, 'Medium': 1, 'High': 2, '<UNK>': 3}

=== Custom One-Hot Encoder ===
CustomOneHotEncoder(categories=['Chat', 'Email', 'Phone', 'Social media'])
Feature names: ['channel_Chat', 'channel_Email', 'channel_Phone', 'channel_Social media']

=== First 5 Rows (Verification) ===
Ticket Priority  Priority_Encoded Ticket Channel  channel_Chat  channel_Email  channel_Phone  channel_Social m

In [8]:
# ============================================================
# HSRIS — Part 2: Sparse Representation (Keyword Retrieval)
# Built from first principles. No scikit-learn.
#
# WHY SPARSE?
# With ~8,470 documents × 5,000 vocabulary tokens, a dense
# float32 matrix would consume  8470 × 5000 × 4 bytes ≈ 170 MB.
# That sounds manageable, but during computation (TF * IDF,
# intermediate copies, batch ops) the actual peak RAM can be 3-5×
# higher — easily >500 MB just for this one tensor.
# A sparse COO tensor stores only the non-zero entries (typically
# <2 % of cells for text data), slashing memory to ~3-5 MB
# and keeping us well within Kaggle T4 limits.
# ============================================================

import re
import math
import torch
import numpy as np
import pandas as pd
from collections import Counter


# ────────────────────────────────────────────────────────────
# 1. CUSTOM TOKENIZER
# ────────────────────────────────────────────────────────────
def tokenize(text: str) -> list[str]:
    """
    Lowercase the text, strip everything that isn't a letter or
    digit or whitespace, then split on whitespace.
    """
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)   # remove punctuation
    tokens = text.split()
    return tokens


# ────────────────────────────────────────────────────────────
# 2. N-GRAM GENERATOR  (sliding window)
# ────────────────────────────────────────────────────────────
def generate_ngrams(tokens: list[str],
                    ns: tuple[int, ...] = (1, 2, 3)) -> list[str]:
    """
    Produce unigrams, bigrams, and trigrams from a token list
    using a simple sliding window.

    Example
    -------
    >>> generate_ngrams(["the", "cat", "sat"], ns=(1, 2))
    ['the', 'cat', 'sat', 'the_cat', 'cat_sat']
    """
    ngrams: list[str] = []
    for n in ns:
        for i in range(len(tokens) - n + 1):
            ngrams.append("_".join(tokens[i : i + n]))
    return ngrams


# ────────────────────────────────────────────────────────────
# 3. CUSTOM TF-IDF VECTORIZER (→ PyTorch sparse COO tensor)
# ────────────────────────────────────────────────────────────
class CustomTfidfVectorizer:
    """
    Builds a TF-IDF matrix from raw text and outputs a
    **PyTorch sparse COO tensor** — never allocating a dense
    (n_docs × vocab_size) matrix at any point.

    Parameters
    ----------
    max_vocab : int
        Keep only the top-k most frequent n-grams across
        the entire corpus.
    ns : tuple[int, ...]
        Which n-gram sizes to extract (default: 1,2,3).
    """

    def __init__(self, max_vocab: int = 5000,
                 ns: tuple[int, ...] = (1, 2, 3)):
        self.max_vocab = max_vocab
        self.ns = ns

        # Populated during fit()
        self.vocab: dict[str, int] = {}      # token → column index
        self.idf: np.ndarray = np.array([])  # shape (vocab_size,)
        self._is_fitted: bool = False

    # ── fit ──────────────────────────────────────────────────
    def fit(self, documents: pd.Series) -> "CustomTfidfVectorizer":
        """
        1. Tokenise + n-gram every document.
        2. Build a vocabulary from the top-k most frequent n-grams.
        3. Compute IDF for each vocabulary term.
        """
        n_docs = len(documents)

        # --- Pass 1: count total frequency & document frequency ---
        total_freq: Counter = Counter()      # corpus-wide n-gram counts
        doc_freq: Counter   = Counter()      # in how many docs each n-gram appears

        for doc in documents:
            tokens = tokenize(doc)
            ngrams = generate_ngrams(tokens, self.ns)
            total_freq.update(ngrams)
            # unique n-grams in this document → +1 to doc frequency
            doc_freq.update(set(ngrams))

        # --- Build vocab: top-k by total frequency ---
        most_common = total_freq.most_common(self.max_vocab)
        self.vocab = {term: idx for idx, (term, _) in enumerate(most_common)}

        # --- Compute IDF: log( (1 + N) / (1 + df) ) + 1  (smooth IDF) ---
        vocab_size = len(self.vocab)
        self.idf = np.zeros(vocab_size, dtype=np.float64)
        for term, idx in self.vocab.items():
            df = doc_freq.get(term, 0)
            self.idf[idx] = math.log((1 + n_docs) / (1 + df)) + 1.0

        self._is_fitted = True
        print(f"[TF-IDF fit] Vocabulary size : {vocab_size}")
        print(f"[TF-IDF fit] Total documents : {n_docs}")
        return self

    # ── transform ────────────────────────────────────────────
    def transform(self, documents: pd.Series) -> torch.Tensor:
        """
        Compute TF-IDF scores and return a **sparse COO tensor**.

        Pipeline (per document):
            1. tokenise → n-grams
            2. count term frequencies (TF) within the document
            3. TF(t,d) = count(t,d) / total_ngrams_in_d
            4. TF-IDF(t,d) = TF(t,d) × IDF(t)
            5. L2-normalise the document vector

        We accumulate coordinate lists (row, col, val) across all
        documents and build the sparse tensor in ONE shot — no
        dense intermediate is ever created.
        """
        if not self._is_fitted:
            raise RuntimeError("Vectorizer has not been fitted yet.")

        n_docs = len(documents)
        vocab_size = len(self.vocab)

        # Coordinate lists for COO construction
        row_indices: list[int] = []
        col_indices: list[int] = []
        values:      list[float] = []

        for row_idx, doc in enumerate(documents):
            tokens = tokenize(doc)
            ngrams = generate_ngrams(tokens, self.ns)

            if not ngrams:
                continue  # empty document → no entries

            # Term frequency within this document
            tf_counts: Counter = Counter()
            for ng in ngrams:
                if ng in self.vocab:
                    tf_counts[ng] += 1

            total_ngrams = len(ngrams)  # denominator for TF

            # Compute TF-IDF values for this row
            doc_vals:   list[float] = []
            doc_cols:   list[int]   = []
            for term, count in tf_counts.items():
                col_idx = self.vocab[term]
                tf = count / total_ngrams
                tfidf = tf * self.idf[col_idx]
                doc_vals.append(tfidf)
                doc_cols.append(col_idx)

            # L2 normalisation  (avoid division by zero)
            norm = math.sqrt(sum(v * v for v in doc_vals)) or 1.0
            for c, v in zip(doc_cols, doc_vals):
                row_indices.append(row_idx)
                col_indices.append(c)
                values.append(v / norm)

        # ── Build PyTorch sparse COO tensor in one shot ──
        indices = torch.tensor(
            [row_indices, col_indices], dtype=torch.long
        )
        vals = torch.tensor(values, dtype=torch.float32)
        sparse_tensor = torch.sparse_coo_tensor(
            indices, vals, size=(n_docs, vocab_size)
        )
        # .coalesce() merges any duplicate indices (shouldn't have any
        # here, but good practice for sparse tensors).
        return sparse_tensor.coalesce()

    # ── convenience ──────────────────────────────────────────
    def fit_transform(self, documents: pd.Series) -> torch.Tensor:
        return self.fit(documents).transform(documents)

    def __repr__(self) -> str:
        return (
            f"CustomTfidfVectorizer(max_vocab={self.max_vocab}, "
            f"ns={self.ns}, fitted={self._is_fitted})"
        )


# ────────────────────────────────────────────────────────────
# 4. EXECUTION
# ────────────────────────────────────────────────────────────

# --- Fit-transform on Ticket Description ---
vectorizer = CustomTfidfVectorizer(max_vocab=5000, ns=(1, 2, 3))
tfidf_sparse = vectorizer.fit_transform(df["Ticket Description"])

# --- Summary statistics ---
n_docs, vocab_size = tfidf_sparse.shape
nnz = tfidf_sparse._nnz()                       # number of stored values
total_cells = n_docs * vocab_size
sparsity_pct = (1 - nnz / total_cells) * 100

print(f"\n=== TF-IDF Sparse Tensor ===")
print(f"Shape          : {tfidf_sparse.shape}  (docs × vocab)")
print(f"Non-zero values: {nnz:,}")
print(f"Total cells    : {total_cells:,}")
print(f"Sparsity       : {sparsity_pct:.2f} %")
print(f"Dtype          : {tfidf_sparse.dtype}")
print(f"Layout         : {tfidf_sparse.layout}")
print(f"\nApprox memory  : ~{nnz * 4 / 1024:.1f} KB  (values only)")
print(f"Dense equiv.   : ~{total_cells * 4 / (1024**2):.1f} MB")
print(f"\n↑ The sparse representation stores ONLY the {nnz:,} non-zero")
print(f"  entries instead of all {total_cells:,} cells — saving >95 % RAM")
print(f"  and keeping us safely within Kaggle T4 memory limits.")


[TF-IDF fit] Vocabulary size : 5000
[TF-IDF fit] Total documents : 8469

=== TF-IDF Sparse Tensor ===
Shape          : torch.Size([8469, 5000])  (docs × vocab)
Non-zero values: 846,135
Total cells    : 42,345,000
Sparsity       : 98.00 %
Dtype          : torch.float32
Layout         : torch.sparse_coo

Approx memory  : ~3305.2 KB  (values only)
Dense equiv.   : ~161.5 MB

↑ The sparse representation stores ONLY the 846,135 non-zero
  entries instead of all 42,345,000 cells — saving >95 % RAM
  and keeping us safely within Kaggle T4 memory limits.


In [9]:
# ============================================================
# HSRIS — Part 3: Dense Semantic Layer (Neural Embeddings)
# GloVe 300D + TF-IDF Weighted Sentence Embeddings
# No scikit-learn. Pure PyTorch + NumPy.
# ============================================================

import os
import glob
import torch
import torch.nn as nn
import numpy as np

# ────────────────────────────────────────────────────────────
# 1. LOAD GloVe EMBEDDINGS
# ────────────────────────────────────────────────────────────
GLOVE_DIR = "/kaggle/input/datasets/thanakomsn/glove6b300dtxt"

glove_files = glob.glob(os.path.join(GLOVE_DIR, "**", "*.txt"), recursive=True)
if not glove_files:
    raise FileNotFoundError(f"No .txt file found under {GLOVE_DIR}")
glove_path = glove_files[0]
print(f"[INFO] Loading GloVe from: {glove_path}")


def load_glove(filepath: str) -> dict[str, np.ndarray]:
    """
    Parse a GloVe text file into a dictionary.

    Each line: word 0.1234 -0.5678 ...  (space-separated)
    Returns:  { "word": np.array([0.1234, -0.5678, ...]), ... }
    """
    embeddings: dict[str, np.ndarray] = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            embeddings[word] = vector
    print(f"[GloVe] Loaded {len(embeddings):,} word vectors, "
          f"dim = {next(iter(embeddings.values())).shape[0]}")
    return embeddings


glove_dict = load_glove(glove_path)
EMBED_DIM = next(iter(glove_dict.values())).shape[0]  # 300


# ────────────────────────────────────────────────────────────
# 2. BUILD PyTorch EMBEDDING LAYER  (with <UNK> fallback)
# ────────────────────────────────────────────────────────────
class GloveEmbedding:
    """
    Wraps a pre-trained GloVe dictionary into a torch.nn.Embedding
    layer.  An additional <UNK> vector (random normal) is appended
    so that out-of-vocabulary words map to a valid index instead of
    crashing.
    """

    def __init__(self, glove_dict: dict[str, np.ndarray],
                 embed_dim: int = 300):
        self.embed_dim = embed_dim

        # Word → index mapping  (reserve last index for <UNK>)
        words = list(glove_dict.keys())
        self.word_to_idx: dict[str, int] = {
            w: i for i, w in enumerate(words)
        }
        self.unk_idx = len(words)
        self.word_to_idx["<UNK>"] = self.unk_idx

        # Stack all GloVe vectors + one random-normal <UNK> vector
        vectors = np.stack(list(glove_dict.values()))            # (V, 300)
        unk_vec = np.random.normal(
            scale=0.6, size=(1, embed_dim)
        ).astype(np.float32)                                      # (1, 300)
        weight_matrix = np.vstack([vectors, unk_vec])             # (V+1, 300)

        # Build a frozen nn.Embedding (no gradient updates needed)
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(weight_matrix), freeze=True
        )
        print(f"[Embedding] Vocab size (incl. <UNK>): "
              f"{self.embedding.num_embeddings:,}")
        print(f"[Embedding] Embedding dim: {self.embed_dim}")

    def get_index(self, word: str) -> int:
        """Return the embedding index; OOV words → <UNK> index."""
        return self.word_to_idx.get(word, self.unk_idx)

    def get_vector(self, word: str) -> torch.Tensor:
        """Return the 300-D embedding vector for a single word."""
        idx = torch.tensor([self.get_index(word)], dtype=torch.long)
        return self.embedding(idx).squeeze(0)


glove_emb = GloveEmbedding(glove_dict, embed_dim=EMBED_DIM)

# Free the raw dict — the nn.Embedding now owns the data
del glove_dict


# ────────────────────────────────────────────────────────────
# 3. TF-IDF WEIGHTED SENTENCE EMBEDDING
# ────────────────────────────────────────────────────────────
def get_sentence_embedding(
    text: str,
    vectorizer,               # fitted CustomTfidfVectorizer from Part 2
    glove: GloveEmbedding,
) -> torch.Tensor:
    """
    Produce a single 300-D sentence vector via IDF-weighted
    averaging of GloVe word embeddings.

    Steps
    -----
    1. Tokenise the text into unigrams (lowercase, no punct).
    2. For each token:
       a. Look up its 300-D GloVe vector (OOV → <UNK> vector).
       b. Look up its IDF score from the fitted TF-IDF vocab.
          If the token isn't in the vocab, default weight = 1.0.
    3. Weighted average = Σ(idf_i · v_i) / Σ(idf_i)

    Returns
    -------
    torch.Tensor of shape (300,)
    """
    # Re-use the tokenizer from Part 2
    tokens = tokenize(text)

    if not tokens:
        # Edge case: empty / non-string → zero vector
        return torch.zeros(glove.embed_dim, dtype=torch.float32)

    vectors:  list[torch.Tensor] = []
    weights:  list[float]        = []

    for word in tokens:
        # --- GloVe vector ---
        vec = glove.get_vector(word)           # shape (300,)

        # --- IDF weight from the fitted vectorizer ---
        if word in vectorizer.vocab:
            idf = float(vectorizer.idf[vectorizer.vocab[word]])
        else:
            idf = 1.0                          # default for unseen tokens

        vectors.append(vec)
        weights.append(idf)

    # Stack into (n_tokens, 300) and (n_tokens,)
    vec_matrix = torch.stack(vectors)           # (T, 300)
    weight_tensor = torch.tensor(weights, dtype=torch.float32)  # (T,)

    # Weighted average: Σ(w_i * v_i) / Σ(w_i)
    weighted_sum = (weight_tensor.unsqueeze(1) * vec_matrix).sum(dim=0)
    embedding = weighted_sum / weight_tensor.sum()

    return embedding


# ────────────────────────────────────────────────────────────
# 4. EXECUTION — process first 5 ticket descriptions
# ────────────────────────────────────────────────────────────
print("\n=== Generating TF-IDF Weighted Sentence Embeddings ===\n")

sample_texts = df["Ticket Description"].head(5).tolist()
embeddings = []

for i, text in enumerate(sample_texts):
    emb = get_sentence_embedding(text, vectorizer, glove_emb)
    embeddings.append(emb)
    print(f"  Row {i}: vector norm = {emb.norm().item():.4f}  "
          f"(first 5 dims: {emb[:5].tolist()})")

# Stack into a single (5, 300) tensor
sentence_embeddings = torch.stack(embeddings)

print(f"\n=== Result ===")
print(f"Shape : {sentence_embeddings.shape}   ✓  (5 documents × 300 dims)")
print(f"Dtype : {sentence_embeddings.dtype}")
print(f"Device: {sentence_embeddings.device}")


[INFO] Loading GloVe from: /kaggle/input/datasets/thanakomsn/glove6b300dtxt/glove.6B.300d.txt
[GloVe] Loaded 400,000 word vectors, dim = 300
[Embedding] Vocab size (incl. <UNK>): 400,001
[Embedding] Embedding dim: 300

=== Generating TF-IDF Weighted Sentence Embeddings ===

  Row 0: vector norm = 2.9374  (first 5 dims: [-0.22059009969234467, 0.10335780680179596, -0.02975062094628811, -0.0563284307718277, 0.011895736679434776])
  Row 1: vector norm = 3.1496  (first 5 dims: [-0.1434885710477829, 0.1061321422457695, -0.09077797085046768, -0.12817221879959106, -0.017458435148000717])
  Row 2: vector norm = 3.0453  (first 5 dims: [-0.0567595399916172, 0.04670480638742447, -0.033869002014398575, -0.06672695279121399, -0.019462481141090393])
  Row 3: vector norm = 2.9129  (first 5 dims: [-0.11795412003993988, 0.05818846449255943, 0.07588247209787369, -0.10864581912755966, 0.07028605043888092])
  Row 4: vector norm = 3.1562  (first 5 dims: [-0.0886290892958641, 0.11777651309967041, -0.06155737

In [10]:
# ============================================================
# HSRIS — Part 4: Hybrid Search Logic & Dual GPU Optimization
# All cosine similarity via PyTorch matmul. No sklearn.
# Dual T4 GPUs via torch.nn.DataParallel.
# ============================================================

import time
import torch
import torch.nn as nn
import numpy as np

# ────────────────────────────────────────────────────────────
# 1. PRE-COMPUTE DENSE CORPUS EMBEDDINGS  (GloVe + IDF-weighted)
# ────────────────────────────────────────────────────────────
print("=== Pre-computing Dense Corpus Embeddings ===")

all_descriptions = df["Ticket Description"].tolist()
dense_vectors = []

for i, text in enumerate(all_descriptions):
    emb = get_sentence_embedding(text, vectorizer, glove_emb)
    dense_vectors.append(emb)
    if (i + 1) % 2000 == 0 or (i + 1) == len(all_descriptions):
        print(f"  processed {i + 1:,} / {len(all_descriptions):,} documents")

# Stack into (N, 300)
dense_corpus = torch.stack(dense_vectors)  # (N, 300)

# L2-normalise so cosine similarity = simple dot product
dense_norms = dense_corpus.norm(dim=1, keepdim=True).clamp(min=1e-8)
dense_corpus = dense_corpus / dense_norms

print(f"[Dense corpus] shape: {dense_corpus.shape}, "
      f"dtype: {dense_corpus.dtype}\n")


# ────────────────────────────────────────────────────────────
# 2. L2-NORMALISE THE SPARSE CORPUS
# ────────────────────────────────────────────────────────────
print("=== L2-Normalising Sparse Corpus ===")

# Convert to dense temporarily to compute row-wise norms, then
# move to GPU where ~170 MB is trivial for a T4's 16 GB VRAM.
# We keep the dense version for GPU matmul compatibility with
# DataParallel (which requires regular tensors for input splits).
sparse_dense_corpus = tfidf_sparse.to_dense()  # (N, 5000)

sparse_norms = sparse_dense_corpus.norm(dim=1, keepdim=True).clamp(min=1e-8)
sparse_dense_corpus = sparse_dense_corpus / sparse_norms

print(f"[Sparse corpus (densified for GPU)] shape: "
      f"{sparse_dense_corpus.shape}\n")


# ────────────────────────────────────────────────────────────
# 3. DUAL-GPU SIMILARITY MODULE
# ────────────────────────────────────────────────────────────
class SimilarityScorer(nn.Module):
    """
    Computes cosine similarity between query vectors and the
    full corpus on GPU.  Both sparse (TF-IDF) and dense (GloVe)
    scores are returned.

    Since all vectors are L2-normalised, cosine similarity
    reduces to a simple dot product:
        cos(q, d) = q · d  (when ||q|| = ||d|| = 1)

    The corpus matrices are stored as registered buffers so that
    torch.nn.DataParallel automatically replicates them across
    every available GPU.
    """

    def __init__(self, dense_corpus: torch.Tensor,
                 sparse_corpus: torch.Tensor):
        super().__init__()
        # Buffers are replicated by DataParallel but not trained
        self.register_buffer("dense_corpus", dense_corpus)    # (N, 300)
        self.register_buffer("sparse_corpus", sparse_corpus)  # (N, V)

    def forward(self, query_dense: torch.Tensor,
                query_sparse: torch.Tensor):
        """
        Parameters
        ----------
        query_dense  : (B, 300)  – L2-normalised GloVe embeddings
        query_sparse : (B, V)    – L2-normalised TF-IDF vectors

        Returns
        -------
        dense_scores  : (B, N)   – cosine similarities (dense channel)
        sparse_scores : (B, N)   – cosine similarities (sparse channel)
        """
        # Dot product = cosine sim for unit vectors
        dense_scores = torch.matmul(
            query_dense, self.dense_corpus.t()
        )          # (B, N)
        sparse_scores = torch.matmul(
            query_sparse, self.sparse_corpus.t()
        )        # (B, N)
        return dense_scores, sparse_scores


# --- Instantiate & wrap in DataParallel ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"[Device] Using: {device}  |  GPUs available: {n_gpus}")

scorer = SimilarityScorer(dense_corpus, sparse_dense_corpus)
scorer = scorer.to(device)

if n_gpus > 1:
    scorer = nn.DataParallel(scorer)
    print(f"[DataParallel] Distributing across {n_gpus} GPUs "
          f"({', '.join(torch.cuda.get_device_name(i) for i in range(n_gpus))})")
else:
    print(f"[INFO] Running on single device: {device}")

print()


# ────────────────────────────────────────────────────────────
# 4. QUERY ENCODER — convert raw text into GPU-ready vectors
# ────────────────────────────────────────────────────────────
def encode_queries(texts: list[str],
                   vectorizer_obj,
                   glove: GloveEmbedding,
                   device: torch.device):
    """
    Convert a batch of raw query strings into L2-normalised
    dense and sparse vectors on the target device.

    Returns
    -------
    q_dense  : (B, 300)  float32 tensor on `device`
    q_sparse : (B, V)    float32 tensor on `device`
    """
    # --- Dense: GloVe + IDF-weighted averaging ---
    dense_vecs = []
    for t in texts:
        emb = get_sentence_embedding(t, vectorizer_obj, glove)
        dense_vecs.append(emb)
    q_dense = torch.stack(dense_vecs)                       # (B, 300)
    d_norms = q_dense.norm(dim=1, keepdim=True).clamp(min=1e-8)
    q_dense = q_dense / d_norms

    # --- Sparse: TF-IDF via our custom vectorizer ---
    sparse_tensor = vectorizer_obj.transform(
        pd.Series(texts)
    )                                                         # sparse (B, V)
    q_sparse = sparse_tensor.to_dense()                       # (B, V)
    s_norms = q_sparse.norm(dim=1, keepdim=True).clamp(min=1e-8)
    q_sparse = q_sparse / s_norms

    return q_dense.to(device), q_sparse.to(device)


# ────────────────────────────────────────────────────────────
# 5. HYBRID RETRIEVAL FUNCTION
# ────────────────────────────────────────────────────────────
def hybrid_retrieve(query_text: str,
                    alpha: float = 0.5,
                    top_k: int = 5) -> list[dict]:
    """
    Retrieve the top-k most similar tickets for a query using
    the blended similarity formula:

        Score = α · Dense_Score  +  (1 − α) · Sparse_Score

    Parameters
    ----------
    query_text : str
    alpha      : float in [0, 1], weight for dense (semantic) score
    top_k      : number of results to return

    Returns
    -------
    list of dicts with keys: rank, index, score, subject, description
    """
    q_dense, q_sparse = encode_queries(
        [query_text], vectorizer, glove_emb, device
    )  # each (1, D)

    with torch.no_grad():
        dense_scores, sparse_scores = scorer(q_dense, q_sparse)

    # Blend
    combined = alpha * dense_scores + (1 - alpha) * sparse_scores  # (1, N)
    combined = combined.squeeze(0).cpu()

    # Top-k
    topk_vals, topk_idxs = torch.topk(combined, top_k)

    results = []
    for rank, (idx, score) in enumerate(
        zip(topk_idxs.tolist(), topk_vals.tolist()), start=1
    ):
        results.append({
            "rank": rank,
            "index": idx,
            "score": round(score, 4),
            "subject": df.iloc[idx]["Ticket Subject"],
            "description": df.iloc[idx]["Ticket Description"][:120] + "...",
        })
    return results


# --- Quick demo ---
print("=== Hybrid Retrieval Demo (α=0.5, Top-5) ===\n")
demo_query = df["Ticket Description"].iloc[0]
print(f"Query (first ticket): \"{demo_query[:100]}...\"\n")

for r in hybrid_retrieve(demo_query, alpha=0.5, top_k=5):
    print(f"  #{r['rank']}  [idx={r['index']}]  score={r['score']:.4f}")
    print(f"      {r['subject']}")

print()


# ────────────────────────────────────────────────────────────
# 6. BATCH PERFORMANCE BENCHMARKING
# ────────────────────────────────────────────────────────────
print("=" * 60)
print("  PERFORMANCE BENCHMARK — Dual T4 GPU Pipeline")
print("=" * 60)

# Sample 100 random tickets as the test query set
np.random.seed(42)
test_indices = np.random.choice(len(df), size=100, replace=False)
test_texts = df.iloc[test_indices]["Ticket Description"].tolist()

BATCH_SIZES = [10, 50, 100]
benchmark_results = []

for bs in BATCH_SIZES:
    n_batches = len(test_texts) // bs

    # Pre-encode all queries (include encoding time)
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t_start = time.perf_counter()

    for batch_idx in range(n_batches):
        batch_texts = test_texts[batch_idx * bs : (batch_idx + 1) * bs]

        # Encode queries → GPU tensors
        q_dense, q_sparse = encode_queries(
            batch_texts, vectorizer, glove_emb, device
        )

        # Forward pass through DataParallel scorer
        with torch.no_grad():
            dense_sc, sparse_sc = scorer(q_dense, q_sparse)
            combined = 0.5 * dense_sc + 0.5 * sparse_sc
            _ = torch.topk(combined, 5, dim=1)             # top-5

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t_end = time.perf_counter()

    elapsed = t_end - t_start
    queries_processed = n_batches * bs
    qps = queries_processed / elapsed if elapsed > 0 else float("inf")

    benchmark_results.append({
        "batch_size": bs,
        "queries": queries_processed,
        "time_sec": round(elapsed, 3),
        "queries_per_sec": round(qps, 1),
    })

    print(f"\n  Batch size = {bs:>3d}")
    print(f"    Queries processed : {queries_processed}")
    print(f"    Total time        : {elapsed:.3f} s")
    print(f"    Throughput        : {qps:.1f} queries/sec")

# --- Summary table ---
print("\n" + "=" * 60)
print(f"  {'Batch Size':>10} | {'Queries':>8} | {'Time (s)':>9} | {'QPS':>10}")
print("-" * 60)
for r in benchmark_results:
    print(f"  {r['batch_size']:>10} | {r['queries']:>8} | "
          f"{r['time_sec']:>9.3f} | {r['queries_per_sec']:>10.1f}")
print("=" * 60)

# --- GPU memory summary ---
if torch.cuda.is_available():
    print("\n  GPU Memory Report:")
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024 ** 2)
        resrv = torch.cuda.memory_reserved(i) / (1024 ** 2)
        print(f"    GPU {i} ({torch.cuda.get_device_name(i)}): "
              f"allocated={alloc:.1f} MB, reserved={resrv:.1f} MB")


=== Pre-computing Dense Corpus Embeddings ===
  processed 2,000 / 8,469 documents
  processed 4,000 / 8,469 documents
  processed 6,000 / 8,469 documents
  processed 8,000 / 8,469 documents
  processed 8,469 / 8,469 documents
[Dense corpus] shape: torch.Size([8469, 300]), dtype: torch.float32

=== L2-Normalising Sparse Corpus ===
[Sparse corpus (densified for GPU)] shape: torch.Size([8469, 5000])

[Device] Using: cuda  |  GPUs available: 2
[DataParallel] Distributing across 2 GPUs (Tesla T4, Tesla T4)

=== Hybrid Retrieval Demo (α=0.5, Top-5) ===

Query (first ticket): "I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

..."

  #1  [idx=0]  score=1.0000
      Product setup
  #2  [idx=3414]  score=0.7741
      Delivery problem
  #3  [idx=7752]  score=0.7246
      Data loss
  #4  [idx=94]  score=0.7214
      Account access
  #5  [idx=1081]  score=0.7187
      Battery life

  PERFORMANCE BENCHMARK — Dual T4 GPU Pipeline

  Batch size =  10


In [11]:
# ============================================================
# HSRIS — Part 5: Visualization & Deployment (Gradio UI)
# Interactive hybrid retrieval dashboard for Kaggle Notebooks.
# Assumes all objects from Parts 1–4 are in memory:
#   df, vectorizer, glove_emb, scorer, hybrid_retrieve, device
# ============================================================

import gradio as gr
import pandas as pd

# ────────────────────────────────────────────────────────────
# 1. GRADIO PREDICTION WRAPPER
# ────────────────────────────────────────────────────────────
def predict(query_text: str, alpha: float):
    """
    Wrapper function for Gradio.

    Returns
    -------
    predicted_type : str    – Ticket Type from the #1 blended match
    blended_md     : str    – Markdown table of Top-3 blended results
    tfidf_md       : str    – Markdown table of Top-3 pure keyword matches
    glove_md       : str    – Markdown table of Top-3 pure semantic matches
    """
    if not query_text or not query_text.strip():
        empty = "⚠️ Please enter a ticket description."
        return empty, empty, empty, empty

    # --- Three retrieval passes ---
    keyword_results  = hybrid_retrieve(query_text, alpha=0.0, top_k=3)
    semantic_results = hybrid_retrieve(query_text, alpha=1.0, top_k=3)
    blended_results  = hybrid_retrieve(query_text, alpha=alpha, top_k=3)

    # --- Predicted Ticket Type from #1 blended match ---
    top_idx = blended_results[0]["index"]
    predicted_type = df.iloc[top_idx]["Ticket Type"]

    # --- Format results as Markdown tables ---
    def results_to_markdown(results: list[dict], title: str) -> str:
        lines = [f"### {title}\n"]
        lines.append("| Rank | Score | Subject | Description Preview |")
        lines.append("|:----:|:-----:|:--------|:--------------------|")
        for r in results:
            desc_preview = r["description"][:90].replace("|", "–") + "..."
            subject = r["subject"].replace("|", "–") if r["subject"] else "N/A"
            lines.append(
                f"| {r['rank']} | {r['score']:.4f} | "
                f"{subject} | {desc_preview} |"
            )
        return "\n".join(lines)

    blended_md  = results_to_markdown(
        blended_results,
        f"🔀 Blended Top-3 (α = {alpha:.2f})"
    )
    tfidf_md    = results_to_markdown(
        keyword_results,
        "📝 Pure Keyword (TF-IDF, α = 0.0)"
    )
    glove_md    = results_to_markdown(
        semantic_results,
        "🧠 Pure Semantic (GloVe, α = 1.0)"
    )

    return predicted_type, blended_md, tfidf_md, glove_md


# ────────────────────────────────────────────────────────────
# 2. GRADIO UI LAYOUT
# ────────────────────────────────────────────────────────────
CUSTOM_CSS = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto;
    font-family: 'Inter', 'Segoe UI', sans-serif;
}
.gr-button-primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
}
"""

with gr.Blocks(
    title="HSRIS — Hybrid Semantic Retrieval & Intelligence System",
    css=CUSTOM_CSS,
    theme=gr.themes.Soft(),
) as demo:

    # --- Header ---
    gr.Markdown(
        """
        # 🔎 HSRIS — Hybrid Semantic Retrieval & Intelligence System
        *Dual-channel search combining **TF-IDF keyword matching** with
        **GloVe semantic embeddings**, accelerated on Dual T4 GPUs.*

        ---
        """
    )

    # --- Inputs ---
    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(
                label="📩 Enter a New Ticket Description",
                placeholder="e.g. My laptop screen is flickering and the "
                            "battery drains very fast after the last update...",
                lines=3,
            )
        with gr.Column(scale=1):
            alpha_slider = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.5,
                step=0.05,
                label="⚖️ Alpha (α)",
                info="0 = Pure TF-IDF  |  1 = Pure GloVe",
            )
            search_btn = gr.Button(
                "🚀 Search", variant="primary", size="lg"
            )

    gr.Markdown("---")

    # --- Predicted Ticket Type ---
    predicted_type = gr.Textbox(
        label="🏷️ Predicted Ticket Type (from #1 Blended match)",
        interactive=False,
    )

    # --- Blended Top-3 ---
    blended_output = gr.Markdown(
        label="Blended Results",
    )

    gr.Markdown("---")

    # --- Side-by-side: TF-IDF vs GloVe ---
    gr.Markdown("## 🔬 Channel Comparison: Keyword vs Semantic")
    with gr.Row():
        with gr.Column():
            tfidf_output = gr.Markdown(
                label="TF-IDF Results",
            )
        with gr.Column():
            glove_output = gr.Markdown(
                label="GloVe Results",
            )

    # --- Wire up events ---
    search_btn.click(
        fn=predict,
        inputs=[query_input, alpha_slider],
        outputs=[predicted_type, blended_output, tfidf_output, glove_output],
    )
    query_input.submit(
        fn=predict,
        inputs=[query_input, alpha_slider],
        outputs=[predicted_type, blended_output, tfidf_output, glove_output],
    )

    # --- Footer ---
    gr.Markdown(
        """
        ---
        <center>
        <em>Built from first principles — No scikit-learn •
        Custom TF-IDF (PyTorch Sparse COO) •
        GloVe 300D + IDF-Weighted Embeddings •
        Dual T4 GPU DataParallel</em>
        </center>
        """
    )


# ────────────────────────────────────────────────────────────
# 3. LAUNCH  (share=True for public Kaggle URL)
# ────────────────────────────────────────────────────────────
# share=True generates a temporary public URL (*.gradio.live)
# so you can include the link in your assignment report.
demo.launch(share=True)


/tmp/ipykernel_55/1810230028.py:83: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_55/1810230028.py:83: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://be13d05c0543ff19a5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# ============================================================
# HSRIS — Part 6: Evaluation Metrics for Assignment Report
# Precision@5 + Qualitative GloVe-vs-TF-IDF Examples
# Uses existing: df, hybrid_retrieve, vectorizer, glove_emb, scorer
# ============================================================

import numpy as np

# ────────────────────────────────────────────────────────────
# 1. PRECISION@5  (Ticket Type Matching)
# ────────────────────────────────────────────────────────────
#
# For each query ticket we retrieve the top-5 most similar
# tickets using the hybrid search (α = 0.5).
# A result is a "hit" if its Ticket Type matches the query's
# Ticket Type.  Precision@5 = hits / 5 for each query, then
# we average across all 100 queries.
# ────────────────────────────────────────────────────────────

print("=" * 64)
print("  PRECISION@5 EVALUATION  (Ticket Type Matching, α = 0.5)")
print("=" * 64)

np.random.seed(42)
eval_indices = np.random.choice(len(df), size=100, replace=False)

precision_scores = []

for i, q_idx in enumerate(eval_indices):
    query_text = df.iloc[q_idx]["Ticket Description"]
    true_type  = df.iloc[q_idx]["Ticket Type"]

    results = hybrid_retrieve(query_text, alpha=0.5, top_k=5)

    # Count how many of the 5 retrieved tickets share the same type
    hits = sum(
        1 for r in results
        if df.iloc[r["index"]]["Ticket Type"] == true_type
    )
    p_at_5 = hits / 5.0
    precision_scores.append(p_at_5)

    if (i + 1) % 25 == 0:
        running_avg = np.mean(precision_scores)
        print(f"  [{i+1:>3}/100]  running avg P@5 = {running_avg:.4f}")

avg_precision = np.mean(precision_scores)
std_precision = np.std(precision_scores)

print(f"\n  ┌─────────────────────────────────────────┐")
print(f"  │  Average Precision@5 = {avg_precision:.4f}  (±{std_precision:.4f})  │")
print(f"  └─────────────────────────────────────────┘\n")


# ────────────────────────────────────────────────────────────
# 2. QUALITATIVE EXAMPLES: GloVe wins over TF-IDF
# ────────────────────────────────────────────────────────────
#
# Find cases where:
#   • Semantic search (α=1.0) gets Rank-1 Ticket Type correct
#   • Keyword search  (α=0.0) gets Rank-1 Ticket Type WRONG
#
# These examples showcase the value of semantic understanding
# over pure keyword overlap.
# ────────────────────────────────────────────────────────────

print("=" * 64)
print("  QUALITATIVE EXAMPLES: Semantic (GloVe) > Keyword (TF-IDF)")
print("=" * 64)

# Shuffle all indices so we get diverse examples
np.random.seed(123)
all_indices = np.random.permutation(len(df))

examples_found = []
TARGET_EXAMPLES = 5

for q_idx in all_indices:
    if len(examples_found) >= TARGET_EXAMPLES:
        break

    query_text = df.iloc[int(q_idx)]["Ticket Description"]
    true_type  = df.iloc[int(q_idx)]["Ticket Type"]

    # Pure keyword (TF-IDF only)
    tfidf_results = hybrid_retrieve(query_text, alpha=0.0, top_k=1)
    tfidf_top_idx  = tfidf_results[0]["index"]
    tfidf_top_type = df.iloc[tfidf_top_idx]["Ticket Type"]

    # Pure semantic (GloVe only)
    glove_results = hybrid_retrieve(query_text, alpha=1.0, top_k=1)
    glove_top_idx  = glove_results[0]["index"]
    glove_top_type = df.iloc[glove_top_idx]["Ticket Type"]

    # Keep only cases where GloVe is right AND TF-IDF is wrong
    if glove_top_type == true_type and tfidf_top_type != true_type:
        examples_found.append({
            "q_idx":         int(q_idx),
            "query_snippet": query_text[:120],
            "true_type":     true_type,
            "tfidf_idx":     tfidf_top_idx,
            "tfidf_snippet": df.iloc[tfidf_top_idx]["Ticket Description"][:120],
            "tfidf_type":    tfidf_top_type,
            "glove_idx":     glove_top_idx,
            "glove_snippet": df.iloc[glove_top_idx]["Ticket Description"][:120],
            "glove_type":    glove_top_type,
        })

# --- Pretty-print the examples ---
for num, ex in enumerate(examples_found, start=1):
    print(f"\n{'─' * 64}")
    print(f"  Example {num}")
    print(f"{'─' * 64}")

    print(f"  📩 QUERY (idx={ex['q_idx']}):")
    print(f"     \"{ex['query_snippet']}...\"")
    print(f"     True Ticket Type: ✅ {ex['true_type']}")

    print(f"\n  📝 TF-IDF Rank-1 (idx={ex['tfidf_idx']}):")
    print(f"     \"{ex['tfidf_snippet']}...\"")
    print(f"     Ticket Type: ❌ {ex['tfidf_type']}")

    print(f"\n  🧠 GloVe Rank-1 (idx={ex['glove_idx']}):")
    print(f"     \"{ex['glove_snippet']}...\"")
    print(f"     Ticket Type: ✅ {ex['glove_type']}")

print(f"\n{'─' * 64}")
print(f"  Found {len(examples_found)} examples where semantic search")
print(f"  correctly identified the Ticket Type at Rank 1, while")
print(f"  keyword search failed — demonstrating the value of")
print(f"  dense embeddings for intent-level matching.")
print(f"{'─' * 64}")


  PRECISION@5 EVALUATION  (Ticket Type Matching, α = 0.5)
  [ 25/100]  running avg P@5 = 0.3760
  [ 50/100]  running avg P@5 = 0.3840
  [ 75/100]  running avg P@5 = 0.3813
  [100/100]  running avg P@5 = 0.3920

  ┌─────────────────────────────────────────┐
  │  Average Precision@5 = 0.3920  (±0.1958)  │
  └─────────────────────────────────────────┘

  QUALITATIVE EXAMPLES: Semantic (GloVe) > Keyword (TF-IDF)

────────────────────────────────────────────────────────────────
  Example 1
────────────────────────────────────────────────────────────────
  📩 QUERY (idx=4029):
     "I'm having an issue with the {product_purchased}. Please assist.

- - - - - - - - - - - - - - - - - - - - - -

Tables I'..."
     True Ticket Type: ✅ Cancellation request

  📝 TF-IDF Rank-1 (idx=457):
     "I'm having an issue with the {product_purchased}. Please assist. <iid=2B6A7C1a9-A2CC-4F39-9FB7-D4B49D7DA I've noticed th..."
     Ticket Type: ❌ Refund request

  🧠 GloVe Rank-1 (idx=4029):
     "I'm having an 